In [1]:
"""
@property Decorator in Python
===========================

This file is a complete study note + runnable demo about the built-in
@property decorator in Python.

It covers:

1. Why @property exists
2. What @property actually does
3. Getter, setter, and deleter
4. Computed attributes
5. Validation through setters
6. Backward compatibility when changing an API
7. Automatic updates when dependent data changes
8. Common mistakes and edge cases
9. Other important built-in decorators

Run this file directly to see all examples.

------------------------------------------------------------
1) WHY @property EXISTS
------------------------------------------------------------

Sometimes you want a value to look like an attribute:

    person.full_name

but internally you want it to be computed from other values:

    person.first_name + " " + person.last_name

Without @property, you would need a method:

    person.full_name()

That works, but it looks less natural when the value is conceptually a field.

@property lets a method be accessed like an attribute.

This is useful when:
- the value is computed dynamically
- you want read-only access
- you want validation on assignment
- you want to keep an old attribute-style API while changing internals

------------------------------------------------------------
2) WHAT @property ACTUALLY DOES
------------------------------------------------------------

@property turns a method into a managed attribute.

Instead of calling:

    obj.value()

you access:

    obj.value

Under the hood, Python runs the getter method for you.

Important idea:
- it is not a normal stored attribute
- it is a method controlled by the descriptor protocol
- it behaves like a field from the outside

------------------------------------------------------------
3) BASIC PROPERTY EXAMPLE
------------------------------------------------------------
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any


class Student:
    """A student with first name and last name.

    full_name is computed using @property.
    """

    def __init__(self, first_name: str, last_name: str) -> None:
        self.first_name = first_name
        self.last_name = last_name

    @property
    def full_name(self) -> str:
        """Read-only computed attribute."""
        return f"{self.first_name} {self.last_name}"


# ============================================================
# 4) WHY PROPERTY IS BETTER THAN A NORMAL METHOD FOR SOME CASES
# ============================================================

"""
Without property:

    class Student:
        def full_name(self):
            return f"{self.first_name} {self.last_name}"

Then you must write:

    student.full_name()

With property, you write:

    student.full_name

This is cleaner for values that feel like data rather than actions.

Rule of thumb:
- use methods for actions: calculate(), save(), send_email()
- use properties for state-like values: full_name, age, total_price
"""


# ============================================================
# 5) SETTER AND DELETER
# ============================================================

class Employee:
    """Example showing getter, setter, and deleter.

    We store the data internally in a private attribute `_full_name`.
    The property allows external code to interact with it safely.
    """

    def __init__(self, full_name: str) -> None:
        self._full_name = full_name

    @property
    def full_name(self) -> str:
        """Getter: called when you read employee.full_name."""
        print("[getter] full_name accessed")
        return self._full_name

    @full_name.setter
    def full_name(self, value: str) -> None:
        """Setter: called when you assign employee.full_name = ..."""
        print("[setter] full_name assigned")
        if not isinstance(value, str):
            raise TypeError("full_name must be a string")
        if not value.strip():
            raise ValueError("full_name cannot be empty")
        self._full_name = value.strip()

    @full_name.deleter
    def full_name(self) -> None:
        """Deleter: called when you do del employee.full_name."""
        print("[deleter] full_name deleted")
        self._full_name = "<deleted>"


# ============================================================
# 6) PROPERTY FOR VALIDATION
# ============================================================

class BankAccount:
    """A classic use case: validation through property setter."""

    def __init__(self, balance: float) -> None:
        self.balance = balance  # uses the setter

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if not isinstance(value, (int, float)):
            raise TypeError("balance must be a number")
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = float(value)

    def deposit(self, amount: float) -> None:
        self.balance = self.balance + amount

    def withdraw(self, amount: float) -> None:
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance = self.balance - amount


# ============================================================
# 7) AUTOMATIC UPDATES WHEN NAMES CHANGE
# ============================================================

class Person:
    """Demonstrates dependent data.

    email is computed from first_name and last_name, so it updates automatically.
    This answers the question:
    Can methods automatically update when names change?

    Yes, if you make them properties that compute the value from current state.
    """

    def __init__(self, first_name: str, last_name: str) -> None:
        self.first_name = first_name
        self.last_name = last_name

    @property
    def email(self) -> str:
        """Always generated from current names."""
        return f"{self.first_name.lower()}.{self.last_name.lower()}@example.com"

    @property
    def full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

    @full_name.setter
    def full_name(self, value: str) -> None:
        parts = value.strip().split()
        if len(parts) < 2:
            raise ValueError("full_name must contain at least first and last name")
        self.first_name = parts[0]
        self.last_name = " ".join(parts[1:])


# ============================================================
# 8) BACKWARD COMPATIBILITY EXAMPLE
# ============================================================

class Circle:
    """An example of changing an old method-based API to property-based API.

    Suppose earlier code used:
        circle.area()

    Later you want to support:
        circle.area

    @property lets you keep a clean attribute-style API.
    """

    def __init__(self, radius: float) -> None:
        self.radius = radius

    @property
    def area(self) -> float:
        return 3.141592653589793 * self.radius * self.radius

    @property
    def diameter(self) -> float:
        return 2 * self.radius


# ============================================================
# 9) READ-ONLY PROPERTY
# ============================================================

class Temperature:
    """A read-only computed property."""

    def __init__(self, celsius: float) -> None:
        self._celsius = celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @property
    def fahrenheit(self) -> float:
        return (self._celsius * 9 / 5) + 32


# ============================================================
# 10) CUSTOM ERROR HANDLING AROUND PROPERTY USE
# ============================================================

"""
Property methods can raise exceptions just like normal methods.

Examples:
- invalid assignment in setter -> TypeError / ValueError
- trying to delete a property can reset state or raise AttributeError
- accessing a property can fail if required state is missing
"""


class SafeProfile:
    def __init__(self, username: str) -> None:
        self._username = username

    @property
    def username(self) -> str:
        if self._username is None:
            raise AttributeError("username was removed")
        return self._username

    @username.setter
    def username(self, value: str) -> None:
        if not value or not value.strip():
            raise ValueError("username cannot be empty")
        self._username = value.strip()

    @username.deleter
    def username(self) -> None:
        self._username = None


# ============================================================
# 11) COMMON MISTAKES
# ============================================================

"""
Mistake 1: Forgetting to use the same property name
---------------------------------------------------
The setter and deleter must use the same name as the property.

Correct:

    @property
    def age(self): ...

    @age.setter
    def age(self, value): ...

Mistake 2: Recursive property access
------------------------------------
Do not do this inside the getter:

    return self.name

if name is the property itself. That causes infinite recursion.
Use a private attribute like self._name.

Mistake 3: Using property for actions
-------------------------------------
Properties should feel like attributes, not tasks.
Avoid using property for methods that do heavy work or have side effects.

Mistake 4: Assuming property stores data
----------------------------------------
@property does not store anything by itself.
The data is usually stored in a private attribute.

Mistake 5: Confusing attribute access with method calls
--------------------------------------------------------
A property is accessed without parentheses.

    obj.value   # property
    obj.value() # method
"""


# ============================================================
# 12) OTHER IMPORTANT BUILT-IN PYTHON DECORATORS
# ============================================================

"""
There are several built-in decorators in Python:

1. @property
   - turns a method into a managed attribute

2. @staticmethod
   - defines a method that does not use self or cls
   - belongs to the class namespace but behaves like a normal function

3. @classmethod
   - receives the class itself as the first argument (cls)
   - often used for alternate constructors

4. @functools.lru_cache
   - caches function results for faster repeated calls

5. @dataclass
   - automatically generates boilerplate methods for classes

6. @abstractmethod
   - marks a method that must be implemented in child classes

7. @singledispatch
   - registers different implementations for different input types

These are all very common in real Python code.
"""


# ============================================================
# 13) EXAMPLES OF OTHER BUILT-IN DECORATORS
# ============================================================

class MathTools:
    @staticmethod
    def add(a: int, b: int) -> int:
        return a + b

    @classmethod
    def description(cls) -> str:
        return f"This is {cls.__name__}"


@dataclass
class Point:
    x: int
    y: int


# ============================================================
# 14) DIRECT ANSWERS TO THE QUESTIONS
# ============================================================

"""
Q1) What does the property decorator actually do?

@property converts a method into a managed attribute.
You access it like an attribute, but Python runs the getter method behind the scenes.

It is often used for:
- computed values
- validation
- read-only access
- maintaining backward compatibility

Q2) Can methods automatically update when names change?

Yes, if the value is computed from current state using @property.
For example, if email depends on first_name and last_name, then changing the names
will automatically change the email property when it is accessed.

Example:

    person = Person("Rajan", "Raj")
    person.email          # rajan.raj@example.com
    person.first_name = "Aman"
    person.email          # aman.raj@example.com

Q3) Are there other key built-in Python decorators?

Yes. The most important ones are:
- @property
- @staticmethod
- @classmethod
- @dataclass
- @abstractmethod
- @lru_cache

Each one solves a different problem in clean Python design.
"""


# ============================================================
# 15) DEMONSTRATION CODE
# ============================================================

if __name__ == "__main__":
    print("\n========== BASIC PROPERTY ==========")
    student = Student("Rajan", "Raj")
    print("student.full_name =>", student.full_name)

    print("\n========== GETTER / SETTER / DELETER ==========")
    emp = Employee("John Doe")
    print(emp.full_name)
    emp.full_name = "Jane Smith"
    print(emp.full_name)
    del emp.full_name
    print(emp.full_name)

    print("\n========== VALIDATION THROUGH PROPERTY ==========")
    account = BankAccount(500)
    print("Initial balance:", account.balance)
    account.deposit(250)
    print("After deposit:", account.balance)
    account.withdraw(100)
    print("After withdraw:", account.balance)
    try:
        account.balance = -10
    except Exception as e:
        print("Caught:", type(e).__name__, e)

    print("\n========== AUTOMATIC UPDATES ==========")
    person = Person("Rajan", "Raj")
    print("full_name:", person.full_name)
    print("email:", person.email)
    person.first_name = "Aman"
    print("After changing first_name:")
    print("full_name:", person.full_name)
    print("email:", person.email)
    person.full_name = "Arjun Kumar"
    print("After setting full_name:")
    print("first_name:", person.first_name)
    print("last_name:", person.last_name)
    print("email:", person.email)

    print("\n========== BACKWARD COMPATIBILITY STYLE PROPERTY ==========")
    circle = Circle(10)
    print("circle.area =>", circle.area)
    print("circle.diameter =>", circle.diameter)

    print("\n========== READ-ONLY PROPERTY ==========")
    temp = Temperature(25)
    print("temp.celsius =>", temp.celsius)
    print("temp.fahrenheit =>", temp.fahrenheit)

    print("\n========== SAFE PROFILE / DELETER BEHAVIOR ==========")
    profile = SafeProfile("coder123")
    print(profile.username)
    profile.username = "  python_lover  "
    print(profile.username)
    del profile.username
    try:
        print(profile.username)
    except Exception as e:
        print("Caught:", type(e).__name__, e)

    print("\n========== OTHER BUILT-IN DECORATORS ==========")
    print("MathTools.add(2, 3) =>", MathTools.add(2, 3))
    print("MathTools.description() =>", MathTools.description())
    point = Point(2, 5)
    print("Dataclass point =>", point)

    print("\n========== SUMMARY ==========")
    print("@property lets you expose a method like an attribute.")
    print("Setter validates or updates state.")
    print("Deleter handles deletion or cleanup.")
    print("Computed properties automatically reflect the latest data.")



========== BASIC PROPERTY ==========
student.full_name => Rajan Raj

========== GETTER / SETTER / DELETER ==========
[getter] full_name accessed
John Doe
[setter] full_name assigned
[getter] full_name accessed
Jane Smith
[deleter] full_name deleted
[getter] full_name accessed
<deleted>

========== VALIDATION THROUGH PROPERTY ==========
Initial balance: 500.0
After deposit: 750.0
After withdraw: 650.0
Caught: ValueError balance cannot be negative

========== AUTOMATIC UPDATES ==========
full_name: Rajan Raj
email: rajan.raj@example.com
After changing first_name:
full_name: Aman Raj
email: aman.raj@example.com
After setting full_name:
first_name: Arjun
last_name: Kumar
email: arjun.kumar@example.com

========== BACKWARD COMPATIBILITY STYLE PROPERTY ==========
circle.area => 314.1592653589793
circle.diameter => 20

========== READ-ONLY PROPERTY ==========
temp.celsius => 25
temp.fahrenheit => 77.0

========== SAFE PROFILE / DELETER BEHAVIOR ==========
coder123
python_lover
Caught: Attrib